In [41]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import json
from pathlib import Path
import tomllib
from loguru import logger
import csv
import regex as re

# --- Configuration ---
sns.set_theme(style="whitegrid", palette="muted")

In [43]:
# --- Load configuration and data ---
configfile = Path("../config.toml").resolve()
with configfile.open("rb") as f:
    config = tomllib.load(f)

datafile = (Path("..") / Path(config["processed"]) / config["current"]).resolve()
if not datafile.exists():
    logger.warning(
        "Datafile does not exist. Run src/preprocess.py first and check the timestamp!"
    )

df = pd.read_parquet(datafile)
df.columns
df.head()

,timestamp,message,author,has_emoji,is_topk,message_length,has_link,timestamp_category,hour,day_of_week,msg_length
0,2023-02-01 19:59:48+00:00,"Dank voor bericht, Marijke",kooky-tapir,False,False,26,False,evening,19:59:48,2,26
1,2023-02-01 20:46:43+00:00,"Herman goede avond, Mag ik de telefoon nummer...",dynamic-frog,False,False,87,False,evening,20:46:43,2,87
2,2023-02-01 20:52:10+00:00,Voor Ista metertje: 010 2455700,intense-toad,False,True,31,False,evening,20:52:10,2,31
3,2023-02-01 20:52:45+00:00,"Dit gaat dus via Ista, niet via De Combi",intense-toad,False,True,40,False,evening,20:52:45,2,40
4,2023-02-01 20:55:55+00:00,Bedankt Herman 👍🏽🙏🏽,dynamic-frog,True,False,19,False,evening,20:55:55,2,19


In [29]:
# --- Load metadata ---
# meta data (test): nested_users5.json
# meta data (prod): resident_metadata.json

with open("resident_metadata.json", "r") as f:
    nested_users = json.load(f)

author_info_df = (
    pd.DataFrame(nested_users)
    .T
    .reset_index()
    .rename(columns={"index": "author"})
)

# author_info_df.columns
# author_info_df.head()

In [30]:
# --- Merge WhatsApp data with user info ---
df_merged = df.merge(author_info_df, on="author", how="left")

# --- Step 0: Exclude Board_function == "none" ---
df_merged = df_merged[df_merged["Board_function"].str.lower() != "not_applicable"]


# logger.info(f"Merged dataset has {len(df_merged)} messages from {df_merged['author'].nunique()} authors.")
print(f"Merged dataset has {len(df_merged)} messages from {df_merged['author'].nunique()} authors.")

df_merged.columns
# df_merged.head()

Merged dataset has 1678 messages from 9 authors.


Index(['timestamp', 'message', 'author', 'has_emoji', 'is_topk',
       'message_length', 'has_link', 'timestamp_category', 'hour',
       'day_of_week', 'msg_length', 'Name', 'Gender', 'Entrance_nr', 'Unit_nr',
       'Floor_nr', 'Nr_rooms', 'Board_function'],
      dtype='object')

In [31]:
df_merged["word_count"] = (
    df_merged["message"]
    .fillna("")
    .str.split()
    .str.len()
)

df_merged 

,timestamp,message,author,has_emoji,is_topk,message_length,has_link,timestamp_category,hour,day_of_week,msg_length,Name,Gender,Entrance_nr,Unit_nr,Floor_nr,Nr_rooms,Board_function,word_count
2,2023-02-01 20:52:10+00:00,Voor Ista metertje: 010 2455700,intense-toad,False,True,31,False,evening,20:52:10,2,31,H,Male,1,222,8,5,Voorzitter,5
3,2023-02-01 20:52:45+00:00,"Dit gaat dus via Ista, niet via De Combi",intense-toad,False,True,40,False,evening,20:52:45,2,40,H,Male,1,222,8,5,Voorzitter,9
6,2023-02-02 17:05:04+00:00,Ok dat moet je even VvE beheer bellen op via T...,intense-toad,False,True,89,False,worktimes,17:05:04,3,89,H,Male,1,222,8,5,Voorzitter,18
8,2023-02-03 12:45:36+00:00,Beste bewoners Heeft iemand een pakketje aange...,cheerful-boar,False,False,78,False,worktimes,12:45:36,4,78,RF,Male,1,223,8,5,Penningmeester,12
9,2023-02-03 18:20:10+00:00,https://studiodmn.nl/gemist,intense-toad,False,True,27,True,evening,18:20:10,4,27,H,Male,1,222,8,5,Voorzitter,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4551,2025-09-04 17:09:14+00:00,"Zeker, ik geef het door aan energiebesparingsc...",intense-toad,False,True,86,False,worktimes,17:09:14,3,86,H,Male,1,222,8,5,Voorzitter,12
4553,2025-09-05 06:49:41+00:00,De Tocht strippen die tussen de ramen zijn aan...,zany-trout,False,True,214,False,morning,06:49:41,4,214,CC,Male,1,212,5,5,Commissielid 2,43
4554,2025-09-05 08:46:27+00:00,Vandaag worden er 2 planken bij mij in de keuk...,cackling-eland,False,True,176,False,worktimes,08:46:27,4,176,MS,Female,2,241,6,5,Algemeen bestuurslid 1,31
4555,2025-09-05 10:18:05+00:00,En Maya,intense-toad,False,True,7,False,worktimes,10:18:05,4,7,H,Male,1,222,8,5,Voorzitter,2


In [32]:
emoji_pattern = re.compile(r"\p{Emoji}")

df_merged["emoji_count"] = (
    df_merged["message"]
    .fillna("")
    .apply(lambda x: len(emoji_pattern.findall(x)))
)

In [33]:
# write merged data to file as xlsx
# outputfile = Path("merged_whatsapp_userinfo.csv").resolve()
# df_merged.to_csv(outputfile, index=False)

# output = processed / f"{filename}-{now}.csv"

In [34]:
# df_by_author = (
#     df_merged
#     .groupby("author", as_index=False)
#     .agg(
#         message_count=("author", "size"),
#         Board_function=("Board_function", "first"),
#     )
# )


# df_by_author = (
#     df_merged
#     .groupby("author", as_index=False)
#     .agg(
#         message_count=("author", "size"),
#         Board_function=("Board_function", "first"),
#         #total_emojies=("emoji_count", "sum"),
#         messages_with_emojies=("has_emoji", "sum"),
#         total_words=("word_count", "sum")
#     )
# )

df_by_author = (
    df_merged
    .groupby("author", as_index=False)
    .agg(
        message_count=("author", "size"),
        Board_function=("Board_function", "first"),
        messages_with_emojies=("has_emoji", "sum"),
        total_words=("word_count", "sum"),
        total_emojis=("emoji_count", "sum"),
    )
)


In [35]:
# df_by_author.count()
# df_by_author.info()
# df_by_author


In [36]:
# optie 1: percentage messages with emojies
# df_by_author["percentage_messages_with_emojies"] = (
#     df_by_author["messages_with_emojies"]
#     / df_by_author["message_count"]
#     * 100
# )

# optie 2: Emoji’s per 100 woorden 
df_by_author["emojis_per_100_words"] = (
    df_by_author["total_emojis"] / df_by_author["total_words"] * 100
).round(2)


In [37]:
df_by_author["emojis_per_100_words"] = (
    df_by_author["total_emojis"]
    .div(df_by_author["total_words"].replace(0, pd.NA))
    * 100
).round(2)


In [40]:
# df_by_author["percentage_messages_with_emojies"] = (
#     df_by_author["percentage_messages_with_emojies"].round(2)
# )

# df_by_author.sort_values(by="percentage_messages_with_emojies", ascending=False, inplace=True)
# df_by_author

# df_by_author[
#     ["author", "message_count", "total_words", "total_emojis", "emojis_per_100_words"]
# ].sort_values("emojis_per_100_words", ascending=False)

df_by_author.sort_values(by="emojis_per_100_words", ascending=False, inplace=True)
df_by_author


,author,message_count,Board_function,messages_with_emojies,total_words,total_emojis,emojis_per_100_words
4,limber-ostrich,160,Algemeen bestuurslid 3,22,1542,581,37.68
8,zany-trout,108,Commissielid 2,23,1991,604,30.34
6,sudsy-jellyfish,195,Commissielid 3,16,2678,764,28.53
0,cackling-eland,106,Algemeen bestuurslid 1,26,2226,576,25.88
5,peppy-rat,47,Secretaris,19,555,140,25.23
1,cheerful-boar,20,Penningmeester,7,396,82,20.71
7,vivid-rhinoceros,236,Commissielid 1,101,4847,836,17.25
3,intense-toad,782,Voorzitter,42,17863,2713,15.19
2,goofy-tiger,24,Algemeen bestuurslid 2,2,689,87,12.63
